<div dir="rtl" style="text-align: right;">
<h3>1 کتابخانه‌ها و نمودار</h3>
</div>

In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats
from sqlalchemy import URL, create_engine, text
from statsmodels.stats.proportion import proportions_ztest

pio.templates.default = 'plotly_dark'
pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

current_dir = Path.cwd().resolve()

ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "notebooks" / "nba_analysis.ipynb").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Project root was not found. Run the notebook from within the repository."
    )

notebooks_dir = str(ROOT / "notebooks")

if notebooks_dir not in sys.path:
    sys.path.insert(0, notebooks_dir)

import nba_top_players

load_dotenv(ROOT / ".env")

DB_PATH = ROOT / "database" / "data" / "nba_star.db"

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database file was not found: {DB_PATH}")

<div dir="rtl" style="text-align: right;">
<h3>1.1 آزمون فرض</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>2. خواندن داده‌ها</h2>
</div>

In [ ]:
engine = create_engine(
    URL.create("sqlite", database=str(DB_PATH))
)

with engine.connect() as connection:
    integrity_result = connection.execute(
        text("PRAGMA integrity_check")
    ).scalars().all()

    table_names = connection.execute(
        text(
            "SELECT name FROM sqlite_master "
            "WHERE type = 'table' ORDER BY name"
        )
    ).scalars().all()

if integrity_result != ["ok"]:
    raise RuntimeError(f"Database integrity check failed: {integrity_result}")

required_tables = {
    "fact_player_stats",
    "dim_player",
    "dim_season",
    "dim_team_scope",
    "dim_player_context",
    "dim_phase",
}

missing_tables = required_tables - set(table_names)

if missing_tables:
    raise RuntimeError(f"Required tables are missing: {sorted(missing_tables)}")

print(f"Database: {DB_PATH}")
print("Integrity check: OK")
print(f"Tables: {len(table_names)}")

<div dir="rtl" style="text-align: right;">
<h2>3. پاک‌سازی داده‌ها</h2>
<h3>3.1 داده های خالی</h3>

</div>

<div dir="rtl" style="text-align: right;">
<h3>3.2 داده‌های پرت</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>3.3 بازیکنان واجد شرایط و فعال</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>4. شناخت داده‌ها</h2>
<h3>4.1 ویژگی‌های بدنی</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.2 عملکرد بازیکنان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.3 تیم‌ها و قهرمانان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.4 رابطه میان آمار بازیکنان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.5 مقایسه پست‌ها در پنج فصل</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>5. معیار انتخاب بازیکنان برتر</h2>
</div>

<div dir="rtl" style="text-align: right;">

### هدف
برای هر فصل، بازیکنان را با یک **امتیاز واحد** رتبه‌بندی می‌کنیم و 50 نفر اول را انتخاب می‌کنیم.

### چرا یک امتیاز ترکیبی؟
هیچ آمار منفردی عملکرد را کامل نشان نمی‌دهد. برای مثال، مجموع امتیاز به تعداد شوت‌ها وابسته است و کیفیتِ بازی در هر دقیقه را نشان نمی‌دهد. به همین دلیل دو دیدگاه را با هم ترکیب کردیم:

- **شاخص‌های نرخی (کیفیت در هر دقیقه):**
  - **BPM** (Box Plus/Minus): تخمین امتیازی که بازیکن در هر 100 حمله نسبت به یک بازیکن متوسط اضافه می‌کند (صفر = متوسط).
  - **PER** (Player Efficiency Rating): نسبت کارهای مثبت (امتیاز، ریباند، پاس گل، سرقت، بلاک) به کارهای منفی (شوت از دست رفته، توپ‌باخت، خطا) در هر دقیقه؛ میانگین لیگ حدود 15 است.
  - **WS/48** (Win Shares per 48 minutes): سهم بازیکن در بردهای تیم به ازای 48 دقیقه؛ میانگین لیگ حدود 0.1 است.
- **شاخص حجمی (مجموع ارزش در کل فصل):**
  - **WS** (Win Shares): تعداد بردهایی که بازیکن در کل فصل برای تیمش ساخته است. این شاخص باعث می‌شود بازیکنی که نیمی از فصل را غایب بوده، فقط به خاطر نرخ بالا در صدر نیاید.

### مراحل محاسبه
1. فقط **فصل عادی** و **یک ردیف برای هر بازیکن**؛ برای بازیکنانی که وسط فصل تیم عوض کرده‌اند، ردیف مجموعِ همهٔ تیم‌ها (2TM/3TM) استفاده می‌شود.
2. فقط بازیکنان با **حداقل 1000 دقیقه** بازی؛ زیرا آمار نمونه‌های کوچک پرنوسان است و بازیکن با چند دقیقه بازیِ عالی نباید قهرمان شود.
3. **استانداردسازی (z-score)** هر شاخص در همان فصل: 
   $z(x)=\dfrac{x-\mu}{\sigma}$ 
   که $\mu$ و $\sigma$ میانگین و انحراف معیار همان شاخص بین بازیکنان واجد شرایط آن فصل است. این کار شاخص‌هایی با مقیاس‌های متفاوت را قابل جمع می‌کند.
4. محاسبهٔ امتیاز نهایی:

$$\text{Impact}=\frac{z(\text{BPM})+z(\text{PER})+z(\text{WS/48})}{3}$$

$$\text{Score}=0.6\times\text{Impact}+0.4\times z(\text{WS})$$

5. مرتب‌سازی نزولی بر اساس Score و انتخاب 50 نفر اول.

### چرا وزن‌های 0.6 و 0.4؟
کیفیت در هر دقیقه نشانهٔ اصلی توانایی است (وزن بیشتر)، ولی میزان حضور در فصل هم مهم است. این وزن‌ها یک **تصمیم قضاوتی** هستند، نه فرمول «درستِ مطلق»؛ به همین دلیل حساسیت‌سنجی انجام دادیم.


</div>


In [ ]:
top50 = nba_top_players.get_top_players(engine, "2019-20")        # or just pass the path :)))
# all_years = pd.concat([nba_top_players.get_top_players(engine, s) for s in
#                        ["2019-20","2020-21","2021-22","2022-23","2023-24"]])
top50

<div dir="rtl" style="text-align: right;">
<h2>6. آمار توصیفی</h2>
<h3>6.1 قد نامزدهای MVP و بازیکنان برتر</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>6.2 تجربه و قد بازیکنان تیم قهرمان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>6.3 پیشنهاد پوینت‌گارد</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>7. آزمون فرض</h2>
<h3>7.1 تغییر چابکی بازیکنان برتر</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>7.2 تجربه بازیکنان تیم قهرمان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>8. تحلیل‌های تکمیلی</h2>
<h3>8.1 تحلیل تکمیلی اول</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.2 تحلیل تکمیلی دوم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.3 تحلیل تکمیلی سوم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.4 تحلیل تکمیلی چهارم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.5 تحلیل تکمیلی پنجم</h3>
</div>